##### 코랩을 사용할 경우

In [1]:
try:
    # Google Drive를 Colab(/content/drive)rmfja 에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/'내 컴퓨터'/sec06
    %ls
except ImportError:
    pass

Mounted at /google_drive
/google_drive/Othercomputers/내 컴퓨터/sec06
01_dataset.ipynb        03_dataloader.ipynb  05_transform.ipynb
02_dataset_split.ipynb  04_training.ipynb


##### 이전 노트북 실행

In [2]:
%%capture
get_ipython().run_line_magic("run", "03_dataloader.ipynb")

##### 임포트

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

# 재현성(Reproducibility) 설정
torch.manual_seed(42)

##### Device 설정

In [4]:
# GPU(CUDA) 사용 가능 시 'cuda', 아니면 'cpu' 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cuda


##### 모델 구조 정의 및 모델 생성

In [5]:
# 와인 품종 분류 모델 (13개 특징 → 3개 품종)
class WineClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(13, 64),     # 입력: 13개 특징
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 3)       # 출력: 3개 클래스 (품종A / 품종B / 품종C)
        )

    def forward(self, x):
        return self.model(x)

# 모델 생성
model = WineClassifier().to(device)
print("✓ 모델 생성 완료")
print(model)

✓ 모델 생성 완료
WineClassifier(
  (model): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=3, bias=True)
  )
)


##### 손실 함수 및 옵티마이저 생성

In [6]:
# 손실 함수(CrossEntropyLoss) 생성
loss_fn = nn.CrossEntropyLoss()

# Adam 옵티마이저(Optimizer) 생성
optimizer = optim.Adam(model.parameters(), lr=0.001)

##### 훈련하기

In [ ]:
# 에포크 수 설정
epochs = 100

# 조기 종료 설정
# patience: 검증 손실이 개선되지 않아도 허용할 에포크 수
# patience_counter: 개선 없이 누적된 에포크 수
# best_val_loss: 현재까지 가장 낮은 검증 손실 (초기값: 무한대)
# best_model_state: 검증 손실이 가장 낮았을 때의 모델 파라미터 복사본
patience         = 10
patience_counter = 0
best_val_loss    = float('inf') # 양의 무한대 (최소값을 찾기 위해 초기값을 크게 설정)
best_model_state = None

# 학습률 스케쥴러 생성
# ReduceLROnPlateau: 검증 손실이 개선되지 않으면 학습률을 자동으로 줄임
# - mode='min'   : 'min': 낮을수록 좋음 (검증 손실 기준) / 'max': 높을수록 좋음 (검증 정확도 기준)
# - factor=0.5   : 개선 없을 때 lr = lr * 0.5 (절반으로 줄임)
# - patience=3   : 3 에포크 동안 개선 없으면 lr 감소
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# 에포크 반복
for epoch in range(epochs):
    # 훈련 단계 -------------------------------------------------------
    model.train()
    batch_loss_sum    = 0
    batch_correct_sum = 0

    # DataLoader가 배치를 자동으로 생성: shuffle=True이므로 매 에포크마다 순서가 섞임
    for batch_x_tensor, batch_y_tensor in train_loader:
        # 1. 데이터를 모델과 같은 device로 이동
        batch_x_tensor = batch_x_tensor.to(device)
        batch_y_tensor = batch_y_tensor.to(device)
        # 2. 그래디언트 초기화
        optimizer.zero_grad()
        # 3. 모델 출력 얻기
        # - 내부적으로 nn.Module.__call__() → forward() 호출
        train_outputs = model(batch_x_tensor)
        # 4. 손실 계산
        loss = loss_fn(train_outputs, batch_y_tensor)
        # 5. 역전파: 그래디언트 계산
        loss.backward()
        # 6. 가중치 업데이트
        optimizer.step()
        # 7. 배치 손실과 맞춘수 누적
        batch_loss_sum    += loss.item()
        batch_correct_sum += (train_outputs.argmax(dim=1) == batch_y_tensor).sum().item()

    # 에포크 손실 및 정확도 계산
    epoch_loss = batch_loss_sum    / len(train_loader)
    epoch_acc  = batch_correct_sum / len(train_dataset) * 100

    # 검증 단계 -------------------------------------------------------
    model.eval()
    val_loss_sum    = 0
    val_correct_sum = 0

    with torch.inference_mode():
        for batch_x_tensor, batch_y_tensor in val_loader:
            # 1. 데이터를 모델과 같은 device로 이동
            batch_x_tensor = batch_x_tensor.to(device)
            batch_y_tensor = batch_y_tensor.to(device)
            # 2. 모델 출력 얻기
            val_outputs      = model(batch_x_tensor)
            # 3. 손실 계산 및 누적
            val_loss_sum    += loss_fn(val_outputs, batch_y_tensor).item()
            # 4. 맞춘 수 누적
            val_correct_sum += (val_outputs.argmax(dim=1) == batch_y_tensor).sum().item()

    # 검증 손실 및 정확도 계산
    val_loss = val_loss_sum    / len(val_loader)
    val_acc  = val_correct_sum / len(val_dataset) * 100

    # 스케줄러: 검증 손실 기준으로 학습률 조정
    lr_scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch+1:3d}/{epochs}]  "
        f"훈련 손실: {epoch_loss:.4f}  훈련 정확도: {epoch_acc:5.1f}%  "
        f"검증 손실: {val_loss:.4f}  검증 정확도: {val_acc:5.1f}%  "
        f"lr: {current_lr:.6f}", end="")

    # 조기 종료 판단
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
        print("  ✓ 최적 모델 저장")
    else:
        patience_counter += 1
        print(f"  (개선 없음 {patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"\n조기 종료: {patience} 에포크 동안 검증 손실 개선 없음")
            break

model.load_state_dict(best_model_state)
print(f"학습 완료  (최적 검증 손실: {best_val_loss:.4f})")

Epoch [  1/100]  훈련 손실: 0.0438  훈련 정확도:  89.4%  검증 손실: 0.0054  검증 정확도: 100.0%  lr: 0.000250  ✓ 최적 모델 저장
Epoch [  2/100]  훈련 손실: 0.0247  훈련 정확도:  90.1%  검증 손실: 0.0052  검증 정확도: 100.0%  lr: 0.000250  ✓ 최적 모델 저장
Epoch [  3/100]  훈련 손실: 0.0230  훈련 정확도:  90.1%  검증 손실: 0.0051  검증 정확도: 100.0%  lr: 0.000250  ✓ 최적 모델 저장
Epoch [  4/100]  훈련 손실: 0.0192  훈련 정확도:  90.1%  검증 손실: 0.0051  검증 정확도: 100.0%  lr: 0.000250  ✓ 최적 모델 저장
Epoch [  5/100]  훈련 손실: 0.0237  훈련 정확도:  90.1%  검증 손실: 0.0050  검증 정확도: 100.0%  lr: 0.000250  ✓ 최적 모델 저장
Epoch [  6/100]  훈련 손실: 0.0270  훈련 정확도:  90.1%  검증 손실: 0.0049  검증 정확도: 100.0%  lr: 0.000250  ✓ 최적 모델 저장
Epoch [  7/100]  훈련 손실: 0.0251  훈련 정확도:  89.4%  검증 손실: 0.0049  검증 정확도: 100.0%  lr: 0.000250  (개선 없음 1/10)
Epoch [  8/100]  훈련 손실: 0.0317  훈련 정확도:  89.4%  검증 손실: 0.0049  검증 정확도: 100.0%  lr: 0.000250  (개선 없음 2/10)
Epoch [  9/100]  훈련 손실: 0.0202  훈련 정확도:  90.1%  검증 손실: 0.0049  검증 정확도: 100.0%  lr: 0.000250  ✓ 최적 모델 저장
Epoch [ 10/100]  훈련 손실: 0.0420  훈련 정확도:  89.4%  검증 손실: 0.004

평가하기

In [8]:
# 평가 모드로 전환
model.eval()

test_loss_sum    = 0
test_correct_sum = 0

# 역전파/기울기 계산없이 테스트셋으로 예측 수행
with torch.inference_mode():
    for batch_x_tensor, batch_y_tensor in test_loader:
        # 1. 데이터를 모델과 같은 device로 이동
        batch_x_tensor = batch_x_tensor.to(device)
        batch_y_tensor = batch_y_tensor.to(device)
        # 2. 모델 출력 얻기
        test_outputs      = model(batch_x_tensor)
        # 3. 손실 계산 및 누적
        test_loss_sum    += loss_fn(test_outputs, batch_y_tensor).item()
        # 4. 맞춘 수 누적
        test_correct_sum += (test_outputs.argmax(dim=1) == batch_y_tensor).sum().item()

# 평가 손실 및 정확도 계산
test_loss     = test_loss_sum    / len(test_loader)
test_accuracy = test_correct_sum / len(test_dataset) * 100

print(f"테스트 손실:   {test_loss:.4f}  ", end="")
print(
    f"테스트 정확도: {test_accuracy:.1f}%  "
    f"({test_correct_sum}/{len(test_dataset)}개 정답)"
)

테스트 손실:   0.2819  테스트 정확도: 94.4%  (17/18개 정답)
